# The Trust Proof

This notebook reconstructs the entire AnchorRegistry from on-chain events alone.

> **No AnchorRegistry API. No Supabase. No account.**
> Just an RPC endpoint and the contract address.
> If AnchorRegistry's servers disappeared tomorrow, this notebook would still work.

> These examples run against Sepolia testnet (`network="sepolia"`).
> To run against Base mainnet: replace `configure(network="sepolia")` with `configure(network="base")`.
> All function calls, record structures, and output shapes are identical across networks.

In [1]:
# Cell 1 — what you need
# - An Ethereum RPC endpoint (Infura, Alchemy, or your own node)
# - The contract address (public on Etherscan forever)
# - The deploy block number
# Nothing else.

from anchorregistry import configure, get_all, to_dataframe
from anchorregistry.constants import CONTRACT_ADDRESS, DEPLOY_BLOCK, RPC_URL

configure(
    network="base-sepolia",
    contract_address="0xb0435faa6deedc1cb6a809008516fe4f4b094f76",
    deploy_block=40223296,
)
print(f"Contract:            {CONTRACT_ADDRESS}")
print(f"Scanning from block: {DEPLOY_BLOCK}")
print(f"RPC endpoint:        {RPC_URL}")

Contract:            TBD
Scanning from block: None


In [2]:
# Cell 2 — reconstruct the full registry
records = get_all()
print(f"Total anchors recovered: {len(records)}")

Total anchors recovered: 5


In [3]:
# Cell 3 — inspect a single record (raw dict)
records[0]

{'ar_id': 'AR-2026-dPXazj6',
 'registered': True,
 'artifact_type_index': 1,
 'artifact_type_name': 'RESEARCH',
 'tx': '0x826f623ce984a9cadeb5d72c710d279e7385cd83ee4815d092e393d5d574ec57',
 'block': 40225238,
 'registrant': '0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44',
 'manifest_hash': '981e16e83f1e0068d87a89e04962b9287bb5f024782beca6fec837304e8c308e',
 'parent_ar_id': '',
 'descriptor': '',
 'title': 'test sepolia anchor',
 'author': '',
 'tree_id': '0xf07140ce4deaf3b5dac859091a079f82e9656f173593feda7895d940b8fa5d13',
 'token_commitment': '0xf07140ce4deaf3b5dac859091a079f82e9656f173593feda7895d940b8fa5d13',
 'data': {'doi': '',
  'institution': '',
  'co_authors': '',
  'url': 'https://anchorregistry.com/'}}

In [4]:
# Cell 4 — load into DataFrame
df = to_dataframe(records)
df.head()

,ar_id,registered,artifact_type_index,artifact_type_name,tx,block,registrant,manifest_hash,parent_ar_id,descriptor,...,data_data_version,data_format,data_row_count,data_schema_url,data_url,code_git_hash,code_license,code_language,code_version,code_url
0,AR-2026-dPXazj6,True,1,RESEARCH,0x826f623ce984a9cadeb5d72c710d279e7385cd83ee48...,40225238,0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44,981e16e83f1e0068d87a89e04962b9287bb5f024782bec...,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AR-2026-86pNXz1,True,2,DATA,0xb050ffe1e882221230d5d2d7c5f68a2eaab9309c065f...,40225277,0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44,e4fde52cce29d5efc0cd0d4a076d410e568937d7d2007f...,AR-2026-dPXazj6,,...,,,,,https://anchorregistry.com/,NaN,NaN,NaN,NaN,NaN
2,AR-2026-yPZBgoP,True,0,CODE,0x4935e22226509ac7bba092a15f01646d252b22ffe242...,40253950,0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44,7befa395c06769bdaf5ddfd3e94a3f79ac6ff67a367dfd...,,,...,NaN,NaN,NaN,NaN,NaN,,,,,
3,AR-2026-A5kgam5,True,0,CODE,0x01c336d30186a283a767d1830bdaf2a8ccecd52142db...,40254559,0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44,40a11cdd88c0df54d1be354dc54e927360376e6080ab76...,,,...,NaN,NaN,NaN,NaN,NaN,,,,,
4,AR-2026-L5RMQQ6,True,0,CODE,0x9532d06aaa4d802b09a387531116641436f42743e4dd...,40254684,0xc7a7afde1177fbf0bb265ea5a616d1b8d7ed8c44,19b09e8ff3e6ae46cfd3c98bc9983ee89d4c319cf4538b...,,,...,NaN,NaN,NaN,NaN,NaN,,,,,


In [5]:
# Cell 5 — explore the registry by artifact type
df.groupby("artifact_type_name").size().sort_values(ascending=False)

artifact_type_name
CODE        3
DATA        1
RESEARCH    1
dtype: int64

In [6]:
# Cell 6 — reconstruct a specific tree
from anchorregistry import get_by_tree

tree = get_by_tree("0xf07140ce4deaf3b5dac859091a079f82e9656f173593feda7895d940b8fa5d13")
for anchor in tree:
    indent = "  " * (1 if anchor["parent_ar_id"] else 0)
    print(f"{indent}{anchor['ar_id']} — {anchor['artifact_type_name']} — {anchor['title']}")

AR-2026-dPXazj6 — RESEARCH — test sepolia anchor


In [7]:
# Cell 7 — verify a specific artifact
from anchorregistry import verify

result = verify("AR-2026-dPXazj6")
print(f"Registered:    {result['registered']}")
print(f"Manifest hash: {result['manifest_hash']}")

Registered:    True
Manifest hash: 981e16e83f1e0068d87a89e04962b9287bb5f024782beca6fec837304e8c308e


---

The output above was generated entirely from public Base event logs.

The contract address is permanent. The deploy block is documented.
Any RPC endpoint works. **The registry cannot be suppressed.**